In [7]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

from pyprojroot import here

# Ajout de la racine au sys.path pour rendre les dossiers importables
root_dir = here()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from building_perceptron.data import fetch_and_save_dataset

In [4]:
# Load Data
output_dir = Path(root_dir) / "data" / "raw_data"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "breast_cancer_wisconsin.csv"

if not output_path.is_file():
    df, X, y = fetch_and_save_dataset(output_path)
else:
    df = pd.read_csv(output_path)
    y = df.iloc[:, -1]
    X = df.iloc[:, :-1]

Note : on peut télécharger le fichier à l'adresse 
https://drive.google.com/file/d/1itXdRo4WJuhqCjtVX4WGvT327WWp4LB7/view?usp=sharing

In [12]:
# Load Data
output_dir = Path(root_dir) / "data" / "raw_data"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "bcw_data.csv"

if not output_path.is_file():
    df, X, y = fetch_and_save_dataset(output_path)
else:
    df = pd.read_csv(output_path)
    y = df.iloc[:, -1]
    X = df.iloc[:, :-1]

# EDA

In [ ]:
import numpy as np
import pandas as pd
from eda_utils import *

### 1. Encodage
On vérifie l'encodage au tout début d'une EDA pour éviter de charger de la donnée corrompue sans s'en rendre compte.
- Si on essaie de lire un fichier encodé en latin-1 ou cp1252 en lui imposant du utf-8, Python plante immédiatement lors du pd.read_csv().
- Parfois, le fichier s'ouvre sans erreur, mais les caractères sont mal interprétés : Montréal devient MontrÃ©al

In [13]:
import unicodedata
import chardet

with open(output_path, 'rb') as file :
    encodage = chardet.detect(file.read(10000))

print(encodage)

{'encoding': 'ascii', 'confidence': 1.0, 'language': 'eo', 'mime_type': 'text/plain'}


In [14]:
# Import du fichier source avec le bon encodage
pd.options.display.max_columns = None
df = pd.read_csv(output_path, encoding='ascii')

In [15]:
df.head(1)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.8,1001.0,0.1184,0.2776,0.3001,0.1471,0.2419,0.07871,1.095,0.9053,8.589,153.4,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.1189


### 

### Analyse des doublons

In [17]:
cols_sans_id = [col for col in df.columns if col != 'id']
nb_doublons = df.duplicated(subset=cols_sans_id).sum()

if nb_doublons == 0:
    print("Aucun doublon trouvé dans le DataFrame.")
else :
    print(f"Nombre de doublons trouvés : {nb_doublons}\nAppelez la fonction duplicate_rows()")

Aucun doublon trouvé dans le DataFrame.


In [ ]:
# Nombre total de lignes en double
# df.duplicated().sum()

# Afficher toutes les lignes impliquées dans un doublon
# df[df.duplicated(keep=False)].sort_values(by=list(df.columns))

# Doublons basés sur une ou plusieurs colonnes
# df[df.duplicated(subset=['id_client'], keep=False)]

# Garder la première occurrence et supprimer les autres
# df_clean = df.drop_duplicates()

# Ou modifier directement le DataFrame existant
# df.drop_duplicates(inplace=True)

### Colonnes vides

In [30]:
# Remplacer les espaces blancs ou chaînes "null" / "None" par np.nan
df_clean = df.replace(r'^\s*$', np.nan, regex=True).replace(['null', 'None', 'nan', 'N/A'], np.nan)

# Tableau récapitulatif des valeurs manquantes
na_summary = pd.DataFrame({
    'Manquants (count)': df_clean.isna().sum(),
    'Manquants (%)': (df_clean.isna().mean() * 100).round(2)
})

na_summary_filtered = na_summary[na_summary['Manquants (count)'] > 0]
if not na_summary_filtered.empty:
    print("Colonnes avec valeurs manquantes :")
    na_summary[na_summary['Manquants (count)'] > 0].sort_values(by='Manquants (%)', ascending=False)
else:
    print("Aucun colonne vide.")


Aucun colonne vide.
